In [0]:
-- Tạo schema--
CREATE SCHEMA IF NOT EXISTS health;

SHOW SCHEMAS IN workspace;

USE SCHEMA health;

CREATE TABLE IF NOT EXISTS demo_patient (
    patient_id STRING,
    name STRING,
    created_at TIMESTAMP
);

INSERT INTO demo_patient VALUES
('P0001', 'Test User', current_timestamp());

SELECT * FROM demo_patient;

--BRONZE LAYER--
USE SCHEMA health;

-- Bronze tables--
SELECT current_catalog(), current_schema();

CREATE TABLE IF NOT EXISTS workspace.health.bronze_patient (
  patient_id   STRING,
  birth_date   DATE,
  gender       STRING,
  region       STRING,
  consent_flag BOOLEAN,
  _ingest_time TIMESTAMP
);

CREATE TABLE IF NOT EXISTS workspace.health.bronze_vitals (
  patient_id   STRING,
  timestamp    TIMESTAMP,
  heart_rate   INT,
  systolic_bp  INT,
  diastolic_bp INT,
  spo2         DOUBLE,
  _ingest_time TIMESTAMP
);

CREATE TABLE IF NOT EXISTS workspace.health.bronze_lab_results (
  patient_id   STRING,
  date         DATE,
  cholesterol  INT,
  glucose      INT,
  hemoglobin   DOUBLE,
  _ingest_time TIMESTAMP
);

--Checking bronze layer--

SHOW TABLES IN workspace.health;

INSERT INTO workspace.health.bronze_patient(patient_id, birth_date, gender, region, consent_flag, _ingest_time)
VALUES ('P0001','1999-06-24','Male','North', true, current_timestamp());

SELECT * FROM workspace.health.bronze_patient LIMIT 5;
--End of Bronze layer--

--SILVER LAYER--
USE CATALOG workspace;
USE SCHEMA health;

CREATE OR REPLACE TABLE silver_patient AS
WITH dedup AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY patient_id ORDER BY _ingest_time DESC) AS rn
  FROM bronze_patient
)
SELECT
  patient_id,
  CAST(birth_date AS DATE) AS birth_date,
  CASE
    WHEN lower(trim(gender)) = 'male'   THEN 'Male'
    WHEN lower(trim(gender)) = 'female' THEN 'Female'
    ELSE 'Unknown'
  END                       AS gender,
  INITCAP(TRIM(region))     AS region,
  CAST(consent_flag AS BOOLEAN) AS consent_flag,
  _ingest_time              AS last_ingest_time
FROM dedup
WHERE rn = 1 AND patient_id IS NOT NULL;
--end--
SELECT current_catalog(), current_schema();

SELECT * FROM workspace.health.silver_patient LIMIT 5;


-- (Tùy chọn) Lưu hàng lỗi để kiểm tra chất lượng dữ liệu
CREATE OR REPLACE TABLE workspace.health.dq_vitals_errors AS
SELECT * 
FROM workspace.health.bronze_vitals
WHERE patient_id IS NULL OR timestamp IS NULL
   OR heart_rate    NOT BETWEEN 40  AND 200
   OR systolic_bp   NOT BETWEEN 80  AND 220
   OR diastolic_bp  NOT BETWEEN 40  AND 140
   OR spo2          NOT BETWEEN 80  AND 100
   OR systolic_bp < diastolic_bp;

-- Tạo bảng Silver sau khi làm sạch
CREATE OR REPLACE TABLE workspace.health.silver_vitals AS
WITH valid AS (
  SELECT *
  FROM workspace.health.bronze_vitals
  WHERE patient_id IS NOT NULL AND timestamp IS NOT NULL
    AND heart_rate   BETWEEN 40 AND 200
    AND systolic_bp  BETWEEN 80 AND 220
    AND diastolic_bp BETWEEN 40 AND 140
    AND spo2         BETWEEN 80 AND 100
    AND systolic_bp >= diastolic_bp
),
dedup AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY patient_id, timestamp ORDER BY _ingest_time DESC) AS rn
  FROM valid
)
SELECT
  patient_id,
  CAST(timestamp AS TIMESTAMP) AS timestamp,
  CAST(heart_rate AS INT)      AS heart_rate,
  CAST(systolic_bp AS INT)     AS systolic_bp,
  CAST(diastolic_bp AS INT)    AS diastolic_bp,
  CAST(spo2 AS DOUBLE)         AS spo2,
  DATE(timestamp)              AS date,
  _ingest_time                 AS last_ingest_time
FROM dedup
WHERE rn = 1;
--end--
--checking--
SELECT * FROM workspace.health.silver_vitals ORDER BY timestamp DESC LIMIT 10;

SELECT 
  MIN(heart_rate) AS min_hr, MAX(heart_rate) AS max_hr,
  MIN(systolic_bp) AS min_sys, MAX(systolic_bp) AS max_sys,
  MIN(diastolic_bp) AS min_dia, MAX(diastolic_bp) AS max_dia,
  MIN(spo2) AS min_spo2, MAX(spo2) AS max_spo2
FROM workspace.health.silver_vitals;

-- count--
SELECT 'bronze_vitals' AS tbl, COUNT(*) AS cnt FROM workspace.health.bronze_vitals
UNION ALL
SELECT 'bronze_patient', COUNT(*) FROM workspace.health.bronze_patient
UNION ALL
SELECT 'silver_vitals', COUNT(*) FROM workspace.health.silver_vitals;

INSERT INTO workspace.health.bronze_vitals
  (patient_id, timestamp, heart_rate, systolic_bp, diastolic_bp, spo2, _ingest_time)
VALUES
  ('P0001', current_timestamp(),        78, 122, 80, 97.5, current_timestamp()),
  ('P0001', date_add(current_timestamp(), -1), 82, 128, 84, 98.2, current_timestamp()),
  ('P0001', date_add(current_timestamp(), -2), 76, 118, 78, 96.8, current_timestamp());

SELECT COUNT(*) FROM workspace.health.silver_vitals;

SELECT * 
FROM workspace.health.silver_vitals
ORDER BY timestamp DESC
LIMIT 10;

SELECT 
  MIN(heart_rate) AS min_hr, MAX(heart_rate) AS max_hr,
  MIN(systolic_bp) AS min_sys, MAX(systolic_bp) AS max_sys,
  MIN(diastolic_bp) AS min_dia, MAX(diastolic_bp) AS max_dia,
  MIN(spo2) AS min_spo2, MAX(spo2) AS max_spo2
FROM workspace.health.silver_vitals;

--Silver lap table--
USE CATALOG workspace;
USE SCHEMA health;

INSERT INTO workspace.health.bronze_lab_results
  (patient_id, date, cholesterol, glucose, hemoglobin, _ingest_time)
VALUES
  ('P0001', current_date(),        185, 102, 14.2, current_timestamp()),
  ('P0001', date_add(current_date(), -1), 210,  95, 13.8, current_timestamp());

-- lưu hàng lỗi --
CREATE OR REPLACE TABLE workspace.health.silver_lab_results AS
WITH valid AS (
  SELECT *
  FROM workspace.health.bronze_lab_results
  WHERE patient_id IS NOT NULL AND date IS NOT NULL
    AND cholesterol BETWEEN 80 AND 400
    AND glucose     BETWEEN 50 AND 400
    AND hemoglobin  BETWEEN 5.0 AND 22.0
),
dedup AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY patient_id, date ORDER BY _ingest_time DESC) AS rn
  FROM valid
)
SELECT
  patient_id,
  CAST(date AS DATE)         AS date,
  CAST(cholesterol AS INT)   AS cholesterol,
  CAST(glucose AS INT)       AS glucose,
  CAST(hemoglobin AS DOUBLE) AS hemoglobin,
  _ingest_time               AS last_ingest_time
FROM dedup
WHERE rn = 1;

SELECT 'silver_patient' AS tbl, COUNT(*) AS cnt FROM workspace.health.silver_patient
UNION ALL
SELECT 'silver_vitals', COUNT(*) FROM workspace.health.silver_vitals
UNION ALL
SELECT 'silver_lab_results', COUNT(*) FROM workspace.health.silver_lab_results;

SELECT * FROM workspace.health.silver_lab_results ORDER BY date DESC LIMIT 5;








